Базовая настройка

In [ ]:
!git clone https://github.com/cyberbazana/dla_dz1.git


Cloning into 'dla_dz1'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 192 (delta 72), reused 182 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (192/192), 54.89 KiB | 349.00 KiB/s, done.
Resolving deltas: 100% (72/72), done.


In [12]:
%cd dla_dz1

calc_WER_CER.py  LICENSE    requirements.txt  train.py
inference.py     README.md  src/


In [13]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.9/859.9 kB 5.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.3/226.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 64.2 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 20.4 MB/s eta 0:00:00
  Attempting uninstall: python-box
    Found existing installation: python-box 7.4.1
    Uninstalling python-box-7.4.1:
      Successfully uninstalled python-box-7.4.1
Note: you ma

In [14]:
%pip install pyctcdecode==0.5.0 --no-deps

Note: you may need to restart the kernel to use updated packages.


Рестарт ненужен.

In [ ]:
!python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='cyberbazana/dz1_dla', filename='model_best.pth', local_dir='./saved/finetune_360_longer')"


model_best.pth: downloading bytes: █████████████████████▉   |  287MB, 24.1MB/s  
model_best.pth: downloading bytes: █████████████████████████|  300MB, 24.9MB/s  
model_best.pth: reconstructing file: 100%|█████████|  328MB /  328MB, 29.8MB/s  


Ячейка ниже запускает инференс финальной модели.

In [43]:
!python inference.py -cn=inference 

Loading the LM will be faster if you build a binary file.
Reading /kaggle/working/dla_dz1/data/lm/3-gram.pruned.3e-7.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
[2026-10-19 19:14:16,725][pyctcdecode.decoder][INFO] - Using arpa instead of binary LM file, decoder instantiation might be slow.
[2026-10-19 19:14:16,818][pyctcdecode.alphabet][INFO] - Alphabet determined to be of regular style.
Loading part test-clean
Preparing librispeech folders: test-clean: 100%|█| 87/87 [00:00<00:00, 189.75it/
Loading part test-other
Preparing librispeech folders: test-other: 100%|█| 90/90 [00:00<00:00, 176.55it/
CTCModel(
  (encoder): ConformerEncoder(
    (layers): ModuleList(
      (0-15): 16 x ConformerBlock(
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ff1): FeedForward(
          (net): Sequential(
       

Если нужно запустить инференс на кастомном датасете, то выполните это:

In [ ]:
!gdown --id "id файла с гугл диска" -O dataset.zip
!unzip -q "dataset.zip"

In [ ]:
!python inference.py -cn=inference inferencer.save_path="ваш путь" datasets=custom_dataset datasets.inference.data_dir="Ваш путь к датасету из условия"

Пример, вы можете создать структуру cool_dir в корне проекта:

```text
cool_dir/
├── audio/
│   ├── UtteranceID1.wav  # may be flac or mp3
│   ├── UtteranceID2.wav
│   └── UtteranceIDn.wav
└── transcriptions/       # ground truth, may not exist (files)
    ├── UtteranceID1.txt
    ├── UtteranceID2.txt
    └── UtteranceIDn.txt
```


Тогда след. команда выполнит инференс

In [ ]:
!python inference.py -cn=inference inferencer.save_path=TEST datasets=custom_dataset datasets.inference.data_dir=./cool_dir

Данные будут сохранены в папке /data/saved/TEST/inference

Если нужно оценить метрики

In [ ]:
!python calc_WER_CER.py --target ./cool_dir/transcriptions --predictions ./data/saved/TEST/inference